# AI Programming — Lecture 10
## Regularization

이 노트북에서는 Lecture 10의 핵심 주제인 **Regularization**을
작은 예제와 Keras 코드로 직접 확인합니다.

이번 강의의 핵심 질문은 다음과 같습니다.

> **Training data에 지나치게 맞추지 않고, 새로운 data에도 잘 동작하게 하려면 어떻게 해야 할까?**

### 학습 목표

실습을 마치면 다음 내용을 설명할 수 있어야 합니다.

- underfitting과 overfitting을 learning curve와 model capacity 관점에서 구분할 수 있습니다.
- regularization과 normalization의 목적 차이를 설명할 수 있습니다.
- L1과 L2 regularization이 parameter에 주는 효과를 비교할 수 있습니다.
- regularization strength $\lambda$가 증가할 때 parameter가 어떻게 변하는지 확인할 수 있습니다.
- L1이 일부 coefficient를 정확히 0으로 만들 수 있음을 확인할 수 있습니다.
- L2 regularization 전에 input scaling이 중요한 이유를 이해합니다.
- L2 regularization과 weight decay의 관계를 설명할 수 있습니다.
- AdamW에서 weight decay를 사용하는 방법을 확인할 수 있습니다.
- Dropout이 training과 inference에서 다르게 동작함을 확인할 수 있습니다.
- image augmentation을 training 과정에 적용할 수 있습니다.
- time-series data에서 간단한 augmentation을 구현할 수 있습니다.

### 실습 방법

1. 셀을 위에서부터 순서대로 실행하세요.
2. 결과 하나만 보기보다 **train/validation gap, coefficient magnitude, sparsity**를 관찰하세요.
3. `TODO`가 표시된 값은 직접 변경해 다시 실행하세요.
4. 이번 실습의 핵심은 **training error를 더 낮추는 것보다 generalization을 개선하는 것**입니다.

## 0. 라이브러리 불러오기

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=5, suppress=True)

# Part I. Overfitting과 Model Capacity

## 1. Train / Validation / Test의 역할

Lecture 10에서는 model selection을 다시 확인합니다.

```text
Train set
→ model parameter 학습

Validation set
→ hyperparameter tuning / model selection

Test set
→ 최종 generalization 성능 평가
```

Validation set을 반복적으로 보면서 hyperparameter를 지나치게 조정하면
validation set에도 overfitting될 수 있습니다.

## 2. Learning Curve에서 Overfitting 찾기

일반적인 overfitting 상황에서는

```text
Training loss   → 계속 감소
Validation loss → 처음에는 감소하다가 다시 증가
```

하는 패턴이 나타납니다.

두 curve 사이의 gap이 커질수록
training data에 비해 validation data에서 성능이 나빠지고 있음을 의미합니다.

In [ ]:
epochs = np.arange(1, 101)

# 개념 설명용 synthetic learning curves
train_loss = 0.9 * np.exp(-epochs / 35) + 0.05
val_loss = (
    0.55 * np.exp(-epochs / 25)
    + 0.12
    + 0.00007 * (epochs - 40) ** 2
)

plt.plot(epochs, train_loss, label="Training loss")
plt.plot(epochs, val_loss, label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Learning Curves")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 확인할 내용

- validation loss가 가장 낮은 시점을 찾아보세요.
- 그 이후에도 training loss는 계속 감소하지만 generalization은 오히려 나빠질 수 있습니다.
- 이런 경우 **early stopping**이 하나의 regularization 방법으로 사용될 수 있습니다.

## 3. Model Capacity: Too Simple vs. Too Complex

Model capacity가 너무 낮으면 중요한 pattern을 학습하지 못하고,
너무 높으면 training data의 noise까지 따라갈 수 있습니다.

간단한 polynomial regression으로 세 경우를 비교합니다.

```text
Too simple  → underfitting
Balanced    → good generalization
Too complex → overfitting
```

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

rng = np.random.default_rng(0)

x_train = np.linspace(-3, 3, 18)
y_train = (
    0.5 * x_train ** 2
    - x_train
    + 2
    + rng.normal(0, 1.2, size=len(x_train))
)

x_grid = np.linspace(-3.2, 3.2, 400)

degrees = [1, 2, 14]

for degree in degrees:
    model = make_pipeline(
        PolynomialFeatures(degree),
        LinearRegression()
    )

    model.fit(
        x_train.reshape(-1, 1),
        y_train
    )

    y_grid = model.predict(
        x_grid.reshape(-1, 1)
    )

    plt.scatter(
        x_train,
        y_train,
        label="Training data"
    )
    plt.plot(
        x_grid,
        y_grid,
        label=f"Degree = {degree}"
    )
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(f"Polynomial Model: Degree {degree}")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

> ### ✅ 체크포인트
>
> 복잡한 model이 항상 더 좋은 것은 아닙니다.
>
> Regularization은 **큰 capacity를 가진 model을 사용하면서도 실제로 사용하는 complexity를 제어**하는 방법으로 볼 수 있습니다.

# Part II. Regularization의 기본 아이디어

## 4. Regularization vs. Normalization

두 용어는 비슷해 보이지만 목적이 다릅니다.

### Regularization

**Generalization 개선**이 목적입니다.

- effective model complexity를 줄임
- overfitting 완화

### Normalization

**Training 안정화**가 목적입니다.

- activation scale 안정화
- optimization을 더 쉽게 만듦

Lecture 9의 normalization과 이번 Lecture 10의 regularization을 구분해서 기억하세요.

## 5. Regularized Objective

일반적인 regularized objective는 다음과 같습니다.

$$
\mathcal{L}_{total}
=
\mathcal{L}_{data}
+
\lambda g(\mathbf{W})
$$

- $\mathcal{L}_{data}$: 원래 task loss
- $g(\mathbf{W})$: regularizer
- $\lambda$: regularization strength

```text
lambda = 0
→ regularization 없음

lambda 증가
→ parameter에 더 강한 제약
```

# Part III. Penalty-Based Regularization

## 6. L1과 L2 Regularization

### L1 Regularization

$$
g(\mathbf{W})
=
\|\mathbf{W}\|_1
=
\sum_i |w_i|
$$

L1은 일부 coefficient를 **정확히 0**으로 만들 수 있습니다.

### L2 Regularization

$$
g(\mathbf{W})
=
\|\mathbf{W}\|_2^2
=
\sum_i w_i^2
$$

L2는 coefficient를 전체적으로 **0 방향으로 shrink**시키는 경향이 있습니다.

## 7. Lecture 예제로 L2 Regularization 효과 확인

Lecture 10의 간단한 데이터를 사용합니다.

| Study Time | 1 | 3 | 5 | 7 |
|---|---:|---:|---:|---:|
| Exam Score | 75 | 77 | 85 | 83 |

모델을 단순하게

$$
\hat{y} = wx + b
$$

라고 하겠습니다.

여기서는 bias $b$는 regularize하지 않고 weight $w$에만 L2 penalty를 적용합니다.

In [ ]:
x = np.array([1, 3, 5, 7], dtype=float)
y = np.array([75, 77, 85, 83], dtype=float)

# Optimization을 쉽게 보기 위해 x를 zero-center
x_centered = x - x.mean()

def objective(w, b, lam):
    y_hat = w * x_centered + b

    data_loss = np.mean(
        (y_hat - y) ** 2
    )

    l2_penalty = lam * (w ** 2)

    return data_loss + l2_penalty

def fit_scalar_l2(lam, lr=0.01, steps=5000):
    w = 0.0
    b = y.mean()

    for _ in range(steps):
        y_hat = w * x_centered + b
        error = y_hat - y

        grad_w = (
            2 * np.mean(error * x_centered)
            + 2 * lam * w
        )

        grad_b = 2 * np.mean(error)

        w -= lr * grad_w
        b -= lr * grad_b

    return w, b

In [ ]:
for lam in [0.0, 0.1, 1.0, 10.0]:
    w, b = fit_scalar_l2(lam)

    print(
        f"lambda={lam:>4} | "
        f"w={w:8.4f} | "
        f"b={b:8.4f}"
    )

### 확인할 내용

$\lambda$가 증가할수록 slope $w$의 절댓값이 작아지는지 확인하세요.

즉:

> **As $\lambda$ increases, the slope shrinks.**

## 8. Lasso vs. Ridge: Coefficient 변화

이번에는 feature가 여러 개인 regression problem을 사용합니다.

실제 target에는 일부 feature만 중요하도록 만들고,
regularization strength를 변화시키면서 coefficient가 어떻게 변하는지 확인합니다.

In [ ]:
from sklearn.datasets import make_regression
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import StandardScaler

X, y_reg, true_coef = make_regression(
    n_samples=200,
    n_features=8,
    n_informative=3,
    noise=15.0,
    coef=True,
    random_state=0
)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("True coefficients:")
print(true_coef)

In [ ]:
alphas = [0.001, 0.01, 0.1, 1.0, 10.0]

print("=== Ridge ===")
for alpha in alphas:
    model = Ridge(alpha=alpha)
    model.fit(X_scaled, y_reg)

    print(
        f"alpha={alpha:<6} "
        f"coef={np.round(model.coef_, 2)}"
    )

print("\n=== Lasso ===")
for alpha in alphas:
    model = Lasso(
        alpha=alpha,
        max_iter=20000
    )
    model.fit(X_scaled, y_reg)

    print(
        f"alpha={alpha:<6} "
        f"coef={np.round(model.coef_, 2)}"
    )

### 확인할 내용

- Ridge: coefficient가 점진적으로 작아집니다.
- Lasso: regularization이 강해지면 일부 coefficient가 정확히 `0`이 될 수 있습니다.

이 때문에 Lasso는 **sparsity / feature selection**이 중요한 경우 유용할 수 있습니다.

### 직접 해보기 1 — Regularization Strength

다음을 추가해 보세요.

```python
alpha = 50
alpha = 100
```

Ridge와 Lasso의 coefficient가 어떻게 달라지는지 비교하세요.

# Part IV. Input Scaling과 L2 Regularization

## 9. 왜 Scaling이 중요한가?

L2 penalty는 coefficient 크기에 직접 적용됩니다.

$$
\lambda \sum_j w_j^2
$$

그런데 feature scale이 크게 다르면
같은 영향력을 표현하기 위해 필요한 coefficient 크기도 크게 달라질 수 있습니다.

예를 들어:

```text
Feature A: 값의 범위 10 ~ 20
Feature B: 값의 범위 약 100,000
```

같은 prediction contribution을 만들더라도 필요한 weight magnitude가 매우 다를 수 있습니다.

따라서 L1/L2 regularization을 사용할 때는
**input feature를 먼저 standardize하는 것이 중요합니다.**

In [ ]:
rng = np.random.default_rng(1)

n = 200

x1 = rng.normal(0, 1, n)
x2 = rng.normal(0, 1000, n)

# 두 feature가 target에 비슷한 규모의 영향을 주도록 설정
y_scale = (
    3.0 * x1
    + 0.003 * x2
    + rng.normal(0, 0.5, n)
)

X_scale = np.column_stack([x1, x2])

ridge_raw = Ridge(alpha=10.0)
ridge_raw.fit(X_scale, y_scale)

scaler = StandardScaler()
X_scale_std = scaler.fit_transform(X_scale)

ridge_scaled = Ridge(alpha=10.0)
ridge_scaled.fit(X_scale_std, y_scale)

print("Without standardization:")
print(ridge_raw.coef_)

print("\nAfter standardization:")
print(ridge_scaled.coef_)

### 확인할 내용

Raw feature에서는 두 coefficient의 numerical scale이 크게 다릅니다.

Standardization 후에는 feature scale이 맞춰지므로
L2 penalty가 feature 간에 더 공정하게 적용됩니다.

# Part V. Bias–Variance Tradeoff

## 10. Regularization Strength와 Bias–Variance

Regularization을 너무 약하게 하면 model이 training data에 민감해져
**variance가 커질 수 있습니다.**

Regularization을 너무 강하게 하면 model이 지나치게 단순해져
**bias가 커질 수 있습니다.**

```text
Weak regularization
→ low bias, high variance 가능

Strong regularization
→ high bias, low variance 가능
```

좋은 generalization을 위해서는 둘 사이의 balance가 필요합니다.

In [ ]:
strength = np.linspace(0, 5, 200)

# 개념 설명용 synthetic curves
variance = 0.05 * np.exp(1.0 * strength)
bias_squared = 2.0 * np.exp(-0.9 * strength)

plt.plot(
    strength,
    bias_squared,
    label="Bias^2"
)
plt.plot(
    strength,
    variance,
    label="Variance"
)
plt.xlabel("Regularization Strength")
plt.ylabel("Relative Magnitude")
plt.title("Bias-Variance Tradeoff")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Part VI. Weight Decay

## 11. L2 Regularization과 Weight Decay

Weight decay는 update 과정에서 weight를 직접 감소시킵니다.

Vanilla SGD에서는 L2 regularization과 weight decay가 같은 형태의 update로 연결될 수 있습니다.

그러나 Adam과 같은 adaptive optimizer에서는
**L2 penalty를 gradient에 포함하는 것과 decoupled weight decay는 일반적으로 같지 않습니다.**

Lecture 10에서는 AdamW를 사용한 decoupled weight decay를 소개합니다.

## 12. AdamW 코드

Lecture slide의 코드를 그대로 사용합니다.

In [ ]:
from tensorflow.keras.optimizers import AdamW

optimizer = AdamW(
    learning_rate=1e-3,
    weight_decay=1e-4
)

print(optimizer)

### Layer-wise L2 Regularization

Layer마다 서로 다른 L2 coefficient를 지정할 수도 있습니다.

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input,
    Dense,
)
from tensorflow.keras.regularizers import l2

model_l2 = Sequential([
    Input(shape=(28 * 28,)),

    Dense(
        512,
        activation="relu",
        kernel_regularizer=l2(0.001)
    ),

    Dense(
        256,
        activation="relu",
        kernel_regularizer=l2(0.005)
    ),

    Dense(
        10,
        activation="softmax"
    )
])

model_l2.summary()

### 확인할 내용

두 방식은 역할이 비슷해 보이지만 적용 위치가 다릅니다.

```text
kernel_regularizer=l2(...)
→ loss에 penalty 항 추가

AdamW(weight_decay=...)
→ optimizer update에서 weight decay를 decouple하여 적용
```

# Part VII. Dropout

## 13. Dropout의 핵심 아이디어

Training 중 hidden unit를 확률적으로 제거합니다.

```text
Training
→ 일부 unit를 random하게 drop

Inference
→ dropout 비활성화
→ 모든 unit 사용
```

특정 neuron에 지나치게 의존하는 것을 줄이고,
여러 feature를 함께 활용하도록 유도합니다.

## 14. NumPy로 Dropout Mask 확인하기

In [ ]:
rng = np.random.default_rng(0)

h = np.array([
    0.5, 1.0, 2.0, 0.2, 1.5, 0.8
])

dropout_rate = 0.5
keep_prob = 1.0 - dropout_rate

mask = (
    rng.random(len(h))
    < keep_prob
).astype(float)

# Inverted dropout
h_dropout = (
    h * mask / keep_prob
)

print("Original h :", h)
print("Mask       :", mask)
print("After dropout:", h_dropout)

Inverted dropout에서는 살아남은 activation을 `1 / keep_prob`만큼 보정합니다.

이렇게 하면 training과 inference에서 activation의 기대 규모를 맞추기 쉽습니다.

## 15. Keras Dropout

Lecture 10의 기본 구조를 그대로 사용합니다.

In [ ]:
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Dense, Dropout

model_dropout = Sequential([
    Input(shape=(100,)),

    Dense(
        512,
        activation="relu"
    ),
    Dropout(0.5),

    Dense(
        256,
        activation="relu"
    ),
    Dropout(0.3),

    Dense(
        10,
        activation="softmax"
    )
])

model_dropout.compile(
    optimizer="adam",
    loss="categorical_crossentropy"
)

model_dropout.summary()

## 16. Training과 Inference에서 Dropout 차이 확인

같은 input을 여러 번 넣어도 `training=True`에서는
dropout mask가 달라지므로 output이 달라질 수 있습니다.

`training=False`에서는 dropout이 비활성화됩니다.

In [ ]:
import tensorflow as tf

tf.random.set_seed(0)

drop_layer = Dropout(0.5)

sample = tf.ones(
    (1, 10),
    dtype=tf.float32
)

print("Training mode:")
for _ in range(3):
    print(
        drop_layer(
            sample,
            training=True
        ).numpy()
    )

print("\nInference mode:")
for _ in range(3):
    print(
        drop_layer(
            sample,
            training=False
        ).numpy()
    )

> ### ✅ 체크포인트
>
> Dropout은 **training-time regularization**입니다.
>
> Inference에서는 random drop을 수행하지 않습니다.

### 직접 해보기 2 — Dropout Rate

```python
Dropout(0.1)
Dropout(0.3)
Dropout(0.5)
Dropout(0.8)
```

rate가 너무 커지면 왜 underfitting이 발생할 수 있는지 생각해 보세요.

# Part VIII. Data Augmentation

## 17. Data Augmentation의 목적

Training sample을 변형하여 더 다양한 입력을 만들어
model이 특정 training example을 단순히 암기하지 않도록 합니다.

Image에서는 다음 transformation을 자주 사용합니다.

```text
Translation
Rotation
Crop
Zoom
Horizontal flip
Color jitter
Blur
Occlusion / Cutout
```

단, augmentation은 **label을 보존하는 범위**에서 적용해야 합니다.

## 18. MNIST Image Augmentation

Lecture 10의 `ImageDataGenerator` 예제를 사용합니다.

MNIST 숫자에서는 horizontal flip을 사용하지 않습니다.

In [ ]:
from tensorflow.keras.datasets import mnist
from tensorflow.keras.preprocessing.image import ImageDataGenerator

(x_train, y_train), (x_test, y_test) = mnist.load_data()

x_train = (
    x_train
    .reshape(-1, 28, 28, 1)
    .astype("float32")
    / 255.0
)

x_test = (
    x_test
    .reshape(-1, 28, 28, 1)
    .astype("float32")
    / 255.0
)

datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=False
)

print("Training shape:", x_train.shape)

## 19. Augmented Image 직접 확인

In [ ]:
sample_x = x_train[:1]
sample_y = y_train[:1]

aug_iter = datagen.flow(
    sample_x,
    sample_y,
    batch_size=1,
    shuffle=False
)

plt.imshow(
    sample_x[0, :, :, 0],
    cmap="gray"
)
plt.title("Original")
plt.axis("off")
plt.show()

for i in range(4):
    aug_x, _ = next(aug_iter)

    plt.imshow(
        aug_x[0, :, :, 0],
        cmap="gray"
    )
    plt.title(f"Augmented {i + 1}")
    plt.axis("off")
    plt.show()

### 확인할 내용

실시간 augmentation을 사용하면 dataset 파일 자체를 늘리지 않고
매 epoch마다 조금씩 다른 training sample을 만들 수 있습니다.

장점:
- 저장 공간 절약
- 더 다양한 sample 제공

단점:
- training 중 augmentation 연산이 추가됨

## 20. Augmentation을 사용한 CNN Training

아래는 Lecture 10의 training 예제입니다.

전체 수업 시간에 따라 `epochs=10` 대신 `epochs=3`으로 먼저 실행해도 됩니다.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense,
    Conv2D,
    MaxPooling2D,
    Flatten,
    Input,
)

model_aug = Sequential([
    Input(shape=(28, 28, 1)),

    Conv2D(
        32,
        (3, 3),
        activation="relu"
    ),

    MaxPooling2D((2, 2)),

    Flatten(),

    Dense(
        128,
        activation="relu"
    ),

    Dense(
        10,
        activation="softmax"
    )
])

model_aug.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model_aug.summary()

In [ ]:
# TODO:
# 전체 실습에서는 epochs=10,
# 빠른 확인은 epochs=3 정도로 실행하세요.

history_aug = model_aug.fit(
    datagen.flow(
        x_train,
        y_train,
        batch_size=32
    ),
    epochs=3,
    validation_data=(
        x_test,
        y_test
    )
)

# Part IX. Time-Series Data Augmentation

## 21. 간단한 Time-Series Augmentation

Lecture 10 마지막에는 time-series augmentation도 소개합니다.

대표적인 방법:

```text
Time shifting
Scaling
Jittering
Time warping
Random masking
Mix-up / synthetic data
```

여기서는 가장 간단한 네 가지를 직접 구현합니다.

In [ ]:
rng = np.random.default_rng(3)

t = np.linspace(0, 4 * np.pi, 120)

series = (
    np.sin(t)
    + 0.2 * np.sin(4 * t)
)

def time_shift(x, shift=15):
    return np.roll(x, shift)

def scale_series(x, factor=1.5):
    return factor * x

def jitter(x, sigma=0.15, seed=0):
    local_rng = np.random.default_rng(seed)
    return (
        x
        + local_rng.normal(
            0,
            sigma,
            size=len(x)
        )
    )

def random_mask(x, start=45, length=20):
    y = x.copy()
    y[start:start + length] = 0.0
    return y

In [ ]:
augmented = {
    "Original": series,
    "Time Shift": time_shift(series),
    "Scaling": scale_series(series),
    "Jittering": jitter(series),
    "Random Masking": random_mask(series),
}

for name, values in augmented.items():
    plt.plot(
        values,
        label=name
    )

plt.xlabel("Time")
plt.ylabel("Value")
plt.title("Time-Series Augmentation")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 주의

Data augmentation은 domain에 따라 label을 바꿀 수 있습니다.

예를 들어:

- 숫자 `6`을 180도 회전하면 `9`처럼 보일 수 있음
- 일부 시계열에서는 time shift가 label 의미를 바꿀 수 있음
- text에서는 단어 치환이 sentiment를 바꿀 수도 있음

따라서 augmentation은 **항상 task의 invariance를 고려해서 선택**해야 합니다.

# 22. 최종 실습

### Overfitting / Capacity

1. polynomial degree를 `1, 2, 5, 14`로 바꾸어 fitted curve를 비교하세요.
2. 너무 복잡한 model이 training data의 noise까지 따라가는지 확인하세요.

### L1 / L2

3. Ridge와 Lasso의 `alpha`를 변경해 coefficient 변화를 확인하세요.
4. Lasso에서 정확히 0이 되는 coefficient 수를 세어 보세요.
5. L2 scalar example에서 $\lambda$를 증가시키고 slope $w$의 변화를 확인하세요.
6. input scaling 전/후 Ridge coefficient를 비교하세요.

### Weight Decay

7. Keras에서 `AdamW(weight_decay=...)`를 만들어 보세요.
8. layer별 `kernel_regularizer=l2(...)`를 서로 다른 값으로 설정해 보세요.

### Dropout

9. `Dropout(0.1)`, `Dropout(0.5)`, `Dropout(0.8)`의 output을 비교하세요.
10. `training=True`와 `training=False`의 차이를 설명하세요.

### Data Augmentation

11. MNIST의 `rotation_range`를 바꾸어 augmented image를 확인하세요.
12. `zoom_range`, `width_shift_range`를 변경해 보세요.
13. time-series jitter의 `sigma`를 변경해 보세요.
14. random masking 길이를 늘렸을 때 pattern이 얼마나 손상되는지 확인하세요.

# 23. 정리

Lecture 10의 regularization 방법을 정리하면 다음과 같습니다.

### Regularization의 목적

```text
Training performance만 높이는 것
        ↓
새로운 data에서도 잘 동작하도록 generalization 개선
```

### Penalty-Based Regularization

| Method | 핵심 효과 |
|---|---|
| L1 | sparsity, 일부 coefficient를 0으로 만들 수 있음 |
| L2 | coefficient를 전체적으로 shrink |
| Weight Decay | update 과정에서 weight magnitude 감소 |

### Training-Based Regularization

| Method | 핵심 효과 |
|---|---|
| Early Stopping | overfitting 시작 전에 training 종료 |
| Dropout | 특정 neuron에 대한 의존 감소 |
| Data Augmentation | training data의 다양성 증가 |

### Bias–Variance

```text
Regularization too weak
→ high variance / overfitting 가능

Regularization too strong
→ high bias / underfitting 가능
```

### 꼭 기억할 것

1. **Regularization은 generalization을 개선하기 위해 model의 effective complexity를 제어합니다.**
2. **L1은 sparsity, L2는 weight shrinkage와 연결됩니다.**
3. **L1/L2를 사용할 때는 feature scaling이 중요합니다.**
4. **Adam에서는 L2 penalty와 decoupled weight decay를 구분해야 합니다.**
5. **Dropout은 training에서만 활성화됩니다.**
6. **Data augmentation은 label을 보존하는 transformation이어야 합니다.**